# Assignment 11 — Before & After Change Log
Documents each change made to `JeremyBarton-Assignment11.ipynb`, showing the original code or text, the replacement, and the reason.

---
## Cell 7 — Image Display Code
**Problem:** `next(iter(train_loader))` pulls a random shuffled batch because the DataLoader was created with `shuffle=True`. Every run shows different images. The assignment requires the first five images in the dataset, which are fixed at indices 0–4 and always have labels `[5, 0, 4, 1, 9]`.

### Before

In [ ]:
# BEFORE — grabs a random batch of 64; shows first 5 of that random batch
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(images[i].squeeze(), cmap="gray")
    ax.set_title(f"Label: {labels[i].item()}")
    ax.axis("off")
plt.show() 

### After

In [ ]:
# AFTER — indexes the dataset directly, bypassing the loader's shuffle entirely
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, ax in enumerate(axes):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"Label: {label}")
    ax.axis("off")
plt.suptitle("First Five Training Images (indices 0–4)")
plt.tight_layout()
plt.show()

**Why it matters:** `train_dataset[i]` returns the sample at a fixed position in the raw dataset. The DataLoader is a wrapper that batches and optionally shuffles — once you go through it, position is lost. Direct indexing is the correct tool when position is what you care about.

---
## Cell 8 — Label Confirmation Text
**Problem:** The original text was observational and vague. It noted that images and labels "appeared to match" after re-running several times, which is trivially true for any DataLoader — labels always travel with their images. It never stated the actual expected labels.

### Before
> *It appears that the first five labels of the dataset match the first five images. Re-ran about 5 times, the labels continue to show the correct number correlation.*

### After
> *The first five training images have labels **[5, 0, 4, 1, 9]**. Because the images are retrieved directly from the dataset by index (not from the shuffled DataLoader), these labels are deterministic and will be the same every run.*

**Why it matters:** The assignment specifies what those labels should be. Stating them explicitly and explaining *why* they are deterministic (direct indexing vs. shuffled loader) demonstrates understanding of the fix, not just the output.

---
## Cell 16 — Training Loop
**Problem:** The original loop had no outer epoch loop, so it made one pass over the 60,000 training images and stopped. A single epoch is not sufficient for a CNN to converge — the first pass mostly establishes rough weight directions. Multiple passes let the optimizer refine those weights. Additionally, training accuracy was never tracked, making the "5% gap" claim in cell 28 unsupported.

### Before

In [ ]:
# BEFORE — single pass, no accuracy tracking
model.train()

for batch_idx, (data, target) in enumerate(train_loader):
    data, target = data.to(device), target.to(device)
    optimizer.zero_grad()
    output = model(data)
    loss = nn.functional.nll_loss(output, target)
    loss.backward()
    optimizer.step()

### After

In [ ]:
# AFTER — 5 epochs, per-epoch loss and accuracy reported
num_epochs = 5

for epoch in range(1, num_epochs + 1):
    model.train()
    train_loss = 0
    correct_train = 0

    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = nn.functional.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * data.size(0)
        correct_train += output.argmax(dim=1).eq(target).sum().item()

    train_loss /= len(train_loader.dataset)
    train_acc = 100. * correct_train / len(train_loader.dataset)
    print(f"Epoch {epoch}/{num_epochs}  —  Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")

**Why it matters:** Each epoch is one full pass over the training set. The model's weights improve incrementally — early epochs make large corrections, later epochs make smaller refinements. Five epochs is a standard baseline for MNIST. Accumulating `train_loss` and `correct_train` inside the loop and averaging at the end of each epoch gives an honest measure of training performance that can be compared to test performance.

---
## Cell 17 — Training Explanation Text
**Problem:** The explanation described a single loop with no mention of epochs or per-epoch metrics. It no longer matched the code after the epoch loop was added.

### Before
> *The for loop iterates over every batch in train_loader. For each batch: (1) Move to device... (2) Zero gradients... (3) Forward pass... (4) Compute loss... (5) Backward pass... (6) Update weights... Then it repeats for the next batch until all 60,000 training images have been seen once.*

### After
> *The outer loop runs the full training set `num_epochs` times, allowing the model to progressively refine its weights across multiple passes. For each epoch and each batch: (1) Move to device... (2) Zero gradients... (3) Forward pass... (4) Compute loss... (5) Backward pass... (6) Update weights... Training loss and accuracy are accumulated across batches and printed at the end of each epoch so progress can be monitored.*

**Why it matters:** Written explanations should accurately describe what the code does. The original ending — "until all 60,000 training images have been seen once" — was now wrong, and the explanation needed to account for the outer epoch loop and the new per-epoch reporting.

---
## Cell 27 — Summary
**Problem:** Two issues. (1) The prose hardcoded `0.0524` as the test loss, but the printed output in the notebook showed `0.0519` — these came from two different runs and the text was never updated. (2) The summary said "trained on just a single epoch," which was accurate before but became incorrect after the epoch loop was added.

### Before
> *...The model performed well with a test loss of **0.0524**... The model correctly classified 9,829 out of 10,000 test images after training on just **a single epoch**. This is strong performance for a single pass through the data and multiple epochs would likely push it higher...*

### After
> *...The model was trained for **5 epochs**, allowing the weights to be refined across multiple full passes through the 60,000 training images. Test loss and accuracy are reported in the cell above...*

**Why it matters:** Hardcoding metric values in prose creates inconsistency the moment the notebook is re-run. Pointing the reader to the printed output cell instead keeps the summary correct regardless of what the new values are. The "single epoch" phrasing also needed to be updated to match the new 5-epoch training code.

---
## Cell 28 — "5% Gap" Claim
**Problem:** The original text claimed a specific 5% gap between training and test accuracy. Training accuracy was never computed anywhere in the notebook, so this number had no source — it was fabricated.

### Before
> *This model performs well at generalizing unseen data. At **5%**, the gap between training and test performance is small. That could be the result of the dropout regularization to prevent overfitting.*

### After
> *The model generalizes well to unseen data. The per-epoch training accuracy printed above can be compared directly to the test accuracy to assess overfitting. Any gap remaining between the two is small, which is consistent with the dropout regularization applied after both the convolutional layers (25%) and the first dense layer (50%) — dropout randomly disables neurons during training, which discourages the model from memorizing training examples and helps it learn more general features.*

**Why it matters:** Every specific claim in a results summary needs to be backed by something computed in the notebook. The 5% figure had no supporting calculation. Now that the epoch loop prints training accuracy, the reader can look at the actual numbers. The revised text makes the argument about dropout without inventing a metric.